# PCOS Detection - Base Paper Replication (SmartScanPCOS Approach)

**Idhu notebook enna pannum:**

Unga core base paper "SmartScanPCOS" (Heliyon, 2024) **99.31% accuracy** achieve pannirukku,
idha approach ah replicate pannrom:

1. **Same dataset** - 541-record Kaggle clinical dataset (Prasoon Kottarathil)
2. **Borderline-SMOTE** - class imbalance handle pannrathukku (364 vs 177 -> balanced ~724)
3. **Mutual Information feature selection** - top 17 features select pannrom (TOMIM-oda replicate)
4. **Two-level Random Forest ensemble** - Level 1: 17 base Random Forest estimators
   trained on selected features, Level 2: another Random Forest trained on Level 1's
   stacked predictions (exact base paper architecture)
5. **8-fold cross-validation, 25 runs** - base paper mathiri statistical validation

**Run panna:** Ovvoru cell-a select pannitu `Shift + Enter` click pannunga.

**IMPORTANT NOTE:** Idha notebook test pannala (sandbox-la dataset illama irundhadhu),
so cell-by-cell run pannitu, edhachum error vandha screenshot podunga, naan udane fix pannuren.


## 1. Libraries Install + Import

In [ ]:
%pip install pandas numpy scikit-learn imbalanced-learn openpyxl --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from imblearn.over_sampling import BorderlineSMOTE
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded successfully!")

## 2. Load Dataset (541-record Kaggle clinical dataset)

In [ ]:
import os

RAW_DIR = "../data/raw"

# Adjust this path if your folder name is different - check what's printed below
CLINICAL_PATH = os.path.join(RAW_DIR, "archive (16)", "PCOS_data_without_infertility.xlsx")

df = pd.read_excel(CLINICAL_PATH, sheet_name="Full_new")
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Dataset shape:", df.shape)
df.head()

## 3. Data Cleaning (matching base paper's preprocessing)

In [ ]:
# Drop identifier columns (base paper does the same - 'Sl. No' and 'Patient File No.' have no predictive value)
drop_cols = ["Sl. No", "Patient File No."]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Rename target column
df = df.rename(columns={"PCOS (Y/N)": "PCOS"})

# Convert everything to numeric where possible
for col in df.columns:
    if col != "PCOS":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Base paper: rows with null values removed for most features;
# features with a limited number of nulls filled with 0
null_counts = df.isnull().sum()
print("Missing values per column (top 10):")
print(null_counts[null_counts > 0].sort_values(ascending=False).head(10))

In [ ]:
# Fill missing values with 0 (matching base paper's approach for sparse nulls)
for col in df.columns:
    if col != "PCOS":
        df[col] = df[col].fillna(0)

df = df.dropna(subset=["PCOS"])
df["PCOS"] = df["PCOS"].astype(int)

print("Final shape after cleaning:", df.shape)
print("\nClass distribution:")
print(df["PCOS"].value_counts())

## 4. Class Balancing with Borderline-SMOTE

Base paper used Borderline-SMOTE to grow the dataset from 541 (364 vs 177) to 724 balanced instances.
We replicate that here.

In [ ]:
X = df.drop(columns=["PCOS"])
y = df["PCOS"]

print("Before SMOTE:", X.shape, "| Class distribution:", y.value_counts().to_dict())

smote = BorderlineSMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("After SMOTE:", X_resampled.shape, "| Class distribution:", pd.Series(y_resampled).value_counts().to_dict())

## 5. Feature Selection - Mutual Information (replicating TOMIM)

Base paper's TOMIM method selected features with MI score above the mean threshold, landing on
17 features. We replicate the same threshold-based selection.

In [ ]:
mi_scores = mutual_info_classif(X_resampled, y_resampled, random_state=42)
mi_series = pd.Series(mi_scores, index=X_resampled.columns).sort_values(ascending=False)

# Threshold = mean of all MI scores (same rule as TOMIM in the base paper)
threshold = mi_series.mean()
selected_features = mi_series[mi_series > threshold].index.tolist()

print(f"MI threshold: {threshold:.4f}")
print(f"Number of features selected: {len(selected_features)}")
print("\nSelected features (ranked):")
print(mi_series[mi_series > threshold])

In [ ]:
plt.figure(figsize=(8, 8))
mi_series[mi_series > threshold].plot(kind="barh", color="#D6336C")
plt.title("Mutual Information Scores - Selected Features")
plt.xlabel("MI Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Two-Level Random Forest Ensemble (Base Paper Architecture)

**Level 1:** A Random Forest with as many trees as selected features (mirroring the base paper's
"17 base estimators" design), trained directly on the selected features.

**Level 2:** A second Random Forest trained on the stacked out-of-fold predictions from Level 1
(this is what makes it a true two-level *stacked* ensemble, avoiding data leakage).

In [ ]:
X_selected = X_resampled[selected_features]
y_final = y_resampled

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_final, test_size=0.2, random_state=42, stratify=y_final
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
from sklearn.model_selection import cross_val_predict

N_ESTIMATORS_L1 = len(selected_features)  # matches base paper's "17 estimators" design

# ---- Level 1: Random Forest, trained with out-of-fold predictions to avoid leakage ----
level1_model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS_L1, max_depth=None, random_state=42
)

# Generate out-of-fold Level-1 predictions on the training set (prevents overfitting into Level 2)
level1_oof_train = cross_val_predict(
    level1_model, X_train, y_train, cv=5, method="predict_proba"
)[:, 1].reshape(-1, 1)

# Fit Level 1 on full training data for use at test time
level1_model.fit(X_train, y_train)
level1_test_pred = level1_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

print("Level 1 training complete.")
print("Level 1 standalone test accuracy:", accuracy_score(y_test, level1_model.predict(X_test)))

In [ ]:
# ---- Level 2: Random Forest trained on Level 1's stacked predictions ----
level2_model = RandomForestClassifier(n_estimators=100, random_state=42)
level2_model.fit(level1_oof_train, y_train)

y_pred = level2_model.predict(level1_test_pred)
y_proba = level2_model.predict_proba(level1_test_pred)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 50)
print("TWO-LEVEL RANDOM FOREST ENSEMBLE - FINAL RESULTS")
print("=" * 50)
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall:    {rec*100:.2f}%")
print(f"F1 Score:  {f1*100:.2f}%")
print()
print(classification_report(y_test, y_pred, target_names=["No PCOS", "PCOS"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="RdPu",
            xticklabels=["No PCOS", "PCOS"], yticklabels=["No PCOS", "PCOS"])
plt.title("Confusion Matrix - Two-Level RF Ensemble")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

## 7. 8-Fold Cross-Validation (Statistical Validation, matching base paper's 25-run protocol)

Base paper ran 8-fold CV for 25 independent runs and reported the average accuracy (99.32%)
with a 95% confidence interval. We replicate this validation protocol here.

In [ ]:
def run_two_level_rf_cv(X, y, n_splits=8, seed=0):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_accuracies = []

    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        l1 = RandomForestClassifier(n_estimators=N_ESTIMATORS_L1, random_state=seed)
        l1_oof = cross_val_predict(l1, X_tr, y_tr, cv=3, method="predict_proba")[:, 1].reshape(-1, 1)
        l1.fit(X_tr, y_tr)
        l1_test = l1.predict_proba(X_te)[:, 1].reshape(-1, 1)

        l2 = RandomForestClassifier(n_estimators=100, random_state=seed)
        l2.fit(l1_oof, y_tr)
        preds = l2.predict(l1_test)

        fold_accuracies.append(accuracy_score(y_te, preds))

    return np.mean(fold_accuracies)

N_RUNS = 25  # matches base paper's 25-run protocol (this cell may take a few minutes)

run_accuracies = []
for run in range(N_RUNS):
    acc = run_two_level_rf_cv(X_selected.reset_index(drop=True), y_final.reset_index(drop=True), n_splits=8, seed=run)
    run_accuracies.append(acc)
    print(f"Run {run+1}/{N_RUNS}: Mean accuracy = {acc*100:.2f}%")

overall_mean = np.mean(run_accuracies) * 100
overall_std = np.std(run_accuracies) * 100
print()
print("=" * 50)
print(f"OVERALL 8-FOLD CV ACCURACY (25 runs): {overall_mean:.2f}% (+/- {overall_std:.2f}%)")
print("=" * 50)
print(f"Base paper (SmartScanPCOS) reported: 99.32%")

## 8. Save the Trained Model

In [ ]:
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(level1_model, "../models/base_paper_level1_rf.pkl")
joblib.dump(level2_model, "../models/base_paper_level2_rf.pkl")
joblib.dump(selected_features, "../models/base_paper_selected_features.pkl")

print("Models saved successfully!")
print(f"\nFINAL SUMMARY:")
print(f"Test set accuracy: {acc*100:.2f}%")
print(f"8-fold CV accuracy (25 runs): {overall_mean:.2f}%")
print(f"Base paper's reported accuracy: 99.31% (test) / 99.32% (CV)")

## Notes

- This replicates the base paper's methodology as closely as possible using the same dataset,
  SMOTE balancing, MI-based feature selection, and two-level Random Forest stacking architecture.
- If your accuracy comes out lower than 99%, this is not unexpected — the base paper's very high
  accuracy is partly a product of SMOTE creating synthetic samples that are easier to separate
  than truly novel real-world cases (a known critique of heavy oversampling on small datasets).
  Report your honest number either way.
- If you want to compare fairly against the base paper in your own report, mention that both use
  the same dataset, SMOTE strategy, and two-level RF architecture with MI-based feature selection.
